<a href="https://colab.research.google.com/github/martinhdezpacheco/tfg-scraping-madrid/blob/main/Base_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''================================================
ATENCIÓN!!!!!! Eliminar antes los archivos de Colab
================================================'''

import requests
from bs4 import BeautifulSoup
import json
import html
import time
import csv
import os
from google.colab import files

def extraer_datos_inmueble(url):
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException:
        return None

    soup = BeautifulSoup(respuesta.text, "html.parser")
    etiqueta = soup.find("estate-show-v2")

    if etiqueta is None:
        return None

    estate_raw = etiqueta.get(":estate")
    if estate_raw is None:
        return None

    estate_json = html.unescape(estate_raw)
    estate = json.loads(estate_json)

    # Sub-objetos anidados donde viven planta, año, ascensor, etc.
    features = estate.get("features") or {}
    energy_data = estate.get("energy_data") or {}

    # "tipo" y "distrito" vienen como diccionarios {"id":..., "title":..., "slug":...}
    # nos quedamos solo con el texto legible ("title")
    tipo_dict = estate.get("type") or {}
    distrito_dict = estate.get("district") or {}

    # m2 viene como string numérico limpio ("62.00"), lo convertimos a float directamente
    m2_raw = estate.get("numeric_surface")
    try:
        m2_valor = float(m2_raw) if m2_raw is not None else None
    except ValueError:
        m2_valor = None

    datos = {
        "id": estate.get("id"),
        "detail_url": url,
        "tipo": tipo_dict.get("title"),
        "distrito": distrito_dict.get("title"),
        "barrio": estate.get("quarter"),
        "precio": estate.get("numeric_price"),
        "m2": m2_valor,
        "habitaciones": estate.get("rooms"),
        "banos": estate.get("bathrooms"),
        "planta": features.get("floor"),
        "anio_construccion": features.get("build_year"),
        "anio_reforma": features.get("renovation_year"),
        "categoria": features.get("category"),
        "ascensor": features.get("elevator"),
        "balcones": features.get("balconies"),
        "terrazas": features.get("terraces"),
        "jardin": features.get("garden"),
        "calefaccion": features.get("heating"),
        "clase_energetica": energy_data.get("class"),
        "points_of_interest": json.dumps(estate.get("points_of_interest")),
        "title": estate.get("title"),
        "description": estate.get("description"),
    }

    # Normalizar strings vacíos a None
    for campo in datos:
        if datos[campo] == "":
            datos[campo] = None

    return datos


In [2]:
'''##########################################################################
BLOQUE 0: Subida de CSVs previos
##########################################################################'''

# Sube URLs_recolectadas.csv obligatoriamente.
# Si ya tienes dataset_final.csv y urls_caidas.csv de ejecuciones anteriores, súbelos también.
# Si es la primera vez que corres la Fase 2, no los tendrás: no pasa nada, el código los crea.
subidos = files.upload()

print("Archivos subidos:", list(subidos.keys()))

Saving dataset_final.csv to dataset_final.csv
Saving urls_caidas.csv to urls_caidas.csv
Saving URLs_recolectadas.csv to URLs_recolectadas.csv
Archivos subidos: ['dataset_final.csv', 'urls_caidas.csv', 'URLs_recolectadas.csv']


In [3]:
'''##########################################################################
BLOQUE 1: Calcular URLs pendientes de procesar
##########################################################################'''

# Cargar todas las URLs recolectadas en Fase 1
todas_las_urls = set()
with open("URLs_recolectadas.csv", newline="", encoding="utf-8-sig") as f:
    lector = csv.DictReader(f)
    for fila in lector:
        todas_las_urls.add(fila["url_detalle"])

print(f"Total de URLs recolectadas: {len(todas_las_urls)}")

# Cargar URLs ya procesadas con éxito (si el archivo existe de una ejecución anterior)
urls_ya_procesadas = set()
if os.path.exists("dataset_final.csv"):
    with open("dataset_final.csv", newline="", encoding="utf-8-sig") as f:
        lector = csv.DictReader(f, delimiter=";")
        for fila in lector:
            urls_ya_procesadas.add(fila["detail_url"])

print(f"URLs ya procesadas con éxito: {len(urls_ya_procesadas)}")

# Cargar URLs que ya sabemos que fallaron (para no reintentarlas)
urls_caidas_previas = set()
if os.path.exists("urls_caidas.csv"):
    with open("urls_caidas.csv", newline="", encoding="utf-8-sig") as f:
        lector = csv.DictReader(f, delimiter=";")
        for fila in lector:
            urls_caidas_previas.add(fila["url"])

print(f"URLs marcadas como caídas previamente: {len(urls_caidas_previas)}")

# URLs que quedan por procesar en esta ejecución
urls_pendientes = todas_las_urls - urls_ya_procesadas - urls_caidas_previas

print(f"URLs pendientes de extraer en esta ejecución: {len(urls_pendientes)}")

Total de URLs recolectadas: 992
URLs ya procesadas con éxito: 979
URLs marcadas como caídas previamente: 3
URLs pendientes de extraer en esta ejecución: 10


In [4]:
'''##########################################################################
BLOQUE 2: Bucle principal con guardado progresivo
##########################################################################'''

# Nombres de los 21 campos que devuelve extraer_datos_inmueble()
campos = ["id", "detail_url", "tipo", "distrito", "barrio", "precio", "m2",
          "habitaciones", "banos", "planta", "anio_construccion", "anio_reforma",
          "categoria", "ascensor", "balcones", "terrazas", "jardin",
          "calefaccion", "clase_energetica", "points_of_interest",
          "title", "description"]

# Si dataset_final.csv no existe todavía, hay que escribir la cabecera primero
escribir_cabecera_dataset = not os.path.exists("dataset_final.csv")
escribir_cabecera_caidas = not os.path.exists("urls_caidas.csv")

contador_ok = 0
contador_fallo = 0

# Abrimos los dos CSVs en modo "a" (append): cada ficha se escribe al momento, no al final
# delimiter=";" para que Excel en español lo abra bien sin descuadrar columnas
with open("dataset_final.csv", "a", newline="", encoding="utf-8-sig") as f_dataset, \
     open("urls_caidas.csv", "a", newline="", encoding="utf-8-sig") as f_caidas:

    escritor_dataset = csv.DictWriter(f_dataset, fieldnames=campos, delimiter=";")
    escritor_caidas = csv.writer(f_caidas, delimiter=";")

    if escribir_cabecera_dataset:
        escritor_dataset.writeheader()
    if escribir_cabecera_caidas:
        escritor_caidas.writerow(["url"])

    for i, url in enumerate(urls_pendientes, start=1):
        datos = extraer_datos_inmueble(url)

        if datos is not None:
            escritor_dataset.writerow(datos)
            contador_ok += 1
        else:
            escritor_caidas.writerow([url])
            contador_fallo += 1

        # Forzamos escritura a disco cada ficha, para no perder nada si Colab se cae
        f_dataset.flush()
        f_caidas.flush()

        if i % 25 == 0:
            print(f"Progreso: {i}/{len(urls_pendientes)} — OK: {contador_ok} — Fallos: {contador_fallo}")

        time.sleep(1.5)  # pausa ética entre peticiones

print(f"\nBucle terminado. Nuevas fichas extraídas: {contador_ok}. Nuevos fallos: {contador_fallo}")


Bucle terminado. Nuevas fichas extraídas: 10. Nuevos fallos: 0


In [5]:
'''##########################################################################
BLOQUE 3: Resumen
##########################################################################'''

total_dataset = sum(1 for _ in open("dataset_final.csv", encoding="utf-8-sig")) - 1  # -1 por la cabecera
total_caidas = sum(1 for _ in open("urls_caidas.csv", encoding="utf-8-sig")) - 1

print(f"Fichas nuevas en esta ejecución: {contador_ok}")
print(f"URLs caídas nuevas en esta ejecución: {contador_fallo}")
print(f"Total acumulado en dataset_final.csv: {total_dataset}")
print(f"Total acumulado en urls_caidas.csv: {total_caidas}")


# Función de diagnóstico: repite la lógica de extraer_datos_inmueble()
# pero en vez de devolver None, indica el motivo exacto del fallo
def diagnosticar_fallo(url):
    try:
        respuesta = requests.get(url, timeout=10)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"ERROR DE RED: {e}"

    soup = BeautifulSoup(respuesta.text, "html.parser")
    etiqueta = soup.find("estate-show-v2")

    if etiqueta is None:
        return "NO SE ENCONTRÓ <estate-show-v2> (¿tipo de anuncio distinto o página vendida con otra plantilla?)"

    estate_raw = etiqueta.get(":estate")
    if estate_raw is None:
        return "LA ETIQUETA EXISTE PERO NO TIENE ATRIBUTO :estate"

    return "Debería haber funcionado (revisa manualmente)"


print("\nURLs caídas y motivo del fallo:")
with open("urls_caidas.csv", newline="", encoding="utf-8-sig") as f:
    lector = csv.DictReader(f, delimiter=";")
    for fila in lector:
        motivo = diagnosticar_fallo(fila["url"])
        print(f"{fila['url']}\n  → {motivo}\n")
        time.sleep(1.5)

Fichas nuevas en esta ejecución: 10
URLs caídas nuevas en esta ejecución: 0
Total acumulado en dataset_final.csv: 989
Total acumulado en urls_caidas.csv: 3

URLs caídas y motivo del fallo:
https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/585198.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/local-comercial/madrid/madrid/585198.html

https://www.tecnocasa.es/venta/piso/madrid/madrid/620645.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/piso/madrid/madrid/620645.html

https://www.tecnocasa.es/venta/piso/madrid/madrid/664142.html
  → ERROR DE RED: 404 Client Error: Not Found for url: https://www.tecnocasa.es/venta/piso/madrid/madrid/664142.html



In [6]:
'''##########################################################################
BLOQUE 4: Descarga de dataset_final.csv para subir a GitHub
##########################################################################'''

files.download("dataset_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
'''##########################################################################
BLOQUE 5: Descarga de urls_caidas.csv para subir a GitHub
##########################################################################'''

files.download("urls_caidas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>